# WildFusion: AnimalCLEF 2025 Pipeline (RTX 5070 Optimized)
This notebook imports the modular Python components and runs the training, validation, and inference cycle for the Jaguar Re-ID challenge.

Modules implemented:
- `models.py`: MegaDescriptor & ALIKED/LightGlue wrappers with FP16 and torch.compile.
- `dataset.py`: PyTorch DataLoader for jaguar re-id.
- `metrics.py`: Identity-balanced mAP calculation.
- `calibration.py`: Isotonic Regression.
- `engine.py`: Training/Validation loop.
- `inference.py`: Fast submission generation with embedding cache and 2-stage verification.


In [ ]:
import pandas as pd
import os
from dataset import get_dataloader
from engine import train_calibration, evaluate_pipeline
from calibration import ScoreCalibrator
from inference import WildFusionInference, generate_submission

# Data paths (from Kaggle)
TRAIN_DIR = "/kaggle/input/jaguar-re-id/train/train"
TEST_DIR = "/kaggle/input/jaguar-re-id/test/test"
TRAIN_CSV = "/kaggle/input/jaguar-re-id/train.csv"
TEST_CSV = "/kaggle/input/jaguar-re-id/test.csv"
SAMPLE_SUB = "/kaggle/input/jaguar-re-id/sample_submission.csv"


### 1. Training Calibration (Score Fusion)
Here we create a DataLoader to generate positive/negative pairs from the training set to fit the Isotonic Regression.


In [ ]:
# 1. Prepare Data
train_loader = get_dataloader(TRAIN_CSV, TRAIN_DIR, num_pairs=500, batch_size=1, num_workers=2)

# 2. Train Calibration
if train_loader:
    calibrator, g_dists, l_scores, labels, identities = train_calibration(train_loader, "calibrator.pkl")
    
    # Evaluate Training mAP
    evaluate_pipeline(calibrator, g_dists, l_scores, labels, identities)
else:
    print("Warning: Skipping training because train_loader is None (likely due to missing CSVs on this environment).")
    calibrator = ScoreCalibrator() # Empty calibrator fallback


### 2. Inference & Submission Generation
Using the trained Calibrator, we instantiate the `WildFusionInference` system. We build the known identities database from the training set, and then predict identities for the test set.


In [ ]:
# 3. Setup Inference System
# Load calibrator if it exists
calibrator.load("calibrator.pkl") if os.path.exists("calibrator.pkl") else print("Using untrained calibrator fallback")

# Initialize system with optimizations (global threshold = 0.85)
system = WildFusionInference(calibrator, novelty_threshold=0.5, top_k=5, global_thresh=0.85)

# Build Database (Embedding cache kicks in here)
system.build_database(TRAIN_CSV, TRAIN_DIR)

# 4. Generate Submission
generate_submission(system, TEST_CSV, TEST_DIR, "submission.csv")
